# RAG Pipeline - Exp1

* Compare LlamaIndex RAG performance in different settings
* Save the configs file for each pipeline

In [3]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)
from groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from datasets import load_dataset
import pandas as pd
import json

from utils import *

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Testing Dataset

In [2]:
dataset = load_dataset("PatronusAI/financebench")

selected_docs =set()
for dct in dataset['train'].select(range(5)):
    selected_docs.add(f'pdfs/{dct["doc_name"]}.pdf')
print(len(selected_docs))

# Load selected pdfs
loaded_pdf = {}
with ThreadPoolExecutor(max_workers=5) as executor:
    loaded_pdf = dict(executor.map(load_pdf_content, list(selected_docs)))
print(f'{len(loaded_pdf)} pdfs loaded')


# Genrate LlamaIndex Documents from each selected item
documents = []
selected_metadata_cols = ['company', 'doc_name', 'doc_period', 'doc_link']

documents = process_items_parallel(
    dataset['train'],
    selected_doc_names=set([selected_doc.replace('pdfs/', '').replace('.pdf', '')
                             for selected_doc in selected_docs]),
    loaded_pdf=loaded_pdf,
    selected_metadata_cols=selected_metadata_cols,
    max_workers=10
)
print(f"Generated {len(documents)} documents")

2
2 pdfs loaded
Generated 5 documents


### Run Pipeline with Different Settings

#### Split Config File --> Per Model Per Config

In [9]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

retriever_params = all_config['retriever_params']
for i in range(len(all_config['yaml_config_files'])):
    yaml_config_file = 'output/saved_configs/' + all_config['yaml_config_files'][i]
    config = {
        'llm_model': all_config['llm_models'][i],
        'embedding_model': all_config['embedding_models'][i],
        'indexing_storage_dir': all_config['indexing_storage_dirs'][i],
        'output_file': all_config['output_files'][i],
        'retriever_params': retriever_params
    }

    with open(yaml_config_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)